In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scienceplots

from FastBEMT import (
    BEMT,
    Environment,
    Propeller,
    Simulation,
    load_propeller_geometry,
)
from FastBEMT.Utils import BladeStressCalculator, Plotter

plt.style.use(["science", "no-latex"])


In [ ]:
rpm = 7000.0
material_density = 2700.0  # kg/m^3

geometry = load_propeller_geometry("Data/10x7E.pkl")
environment = Environment()
simulation = Simulation(
    revolutions=1,
    timesteps_per_revolution=100,
    device="cpu",
)
propeller = Propeller(geometry, environment, simulation)


In [ ]:
bemt = BEMT(propeller, rpm=rpm, v_inf=0.0)
performance = bemt.performance_for()

print(f"Operating point: {rpm:.0f} RPM, hover")
print(f"Thrust: {performance.thrust:.3f} N")
print(f"Torque: {performance.torque:.4f} N m")


In [ ]:
stress_calculator = BladeStressCalculator(propeller)
centrifugal_stress, bending_stress = stress_calculator.compute_stress(
    material_density,
    bemt,
)
combined_stress = centrifugal_stress[:, None] + bending_stress

peak_index = int(np.argmax(np.max(np.abs(combined_stress), axis=1)))
print(f"Peak centrifugal stress: {np.max(np.abs(centrifugal_stress)) / 1e6:.2f} MPa")
print(f"Peak bending stress: {np.max(np.abs(bending_stress)) / 1e6:.2f} MPa")
print(f"Peak combined stress: {np.max(np.abs(combined_stress)) / 1e6:.2f} MPa")
print(f'Peak combined-stress radius: {geometry["r"][peak_index]:.4f} m')


In [ ]:
radius = np.asarray(geometry["r"])
max_bending_stress = np.max(np.abs(bending_stress), axis=1) / 1e6
max_tensile_stress = np.max(combined_stress, axis=1) / 1e6
max_compressive_stress = np.min(combined_stress, axis=1) / 1e6

figure, axis = plt.subplots(figsize=(6.5, 4.0))
axis.plot(radius, centrifugal_stress / 1e6, label="Centrifugal")
axis.plot(radius, max_bending_stress, label="Bending (section maximum)")
axis.plot(radius, max_tensile_stress, label="Maximum combined")
axis.plot(radius, max_compressive_stress, label="Minimum combined")
axis.set_xlabel("Radius [m]")
axis.set_ylabel("Stress [MPa]")
axis.set_title("10x7E stress envelopes at 7000 RPM in hover")
axis.grid(True, linestyle=":")
axis.legend()
figure.tight_layout()
plt.show()


In [ ]:
plotter = Plotter(propeller)
plotter.plot_stress_distribution(
    centrifugal_stress,
    bending_stress,
    figsize=(4.5, 3.0),
    cmap="viridis",
)
